In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "bert-base-uncased"

In [7]:
label2id = {"planner": 0, "delete": 1, "get_event": 2}
id2label = {v: k for k, v in label2id.items()}

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=3,id2label=id2label,label2id=label2id)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2603.31it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [9]:
for name,param in model.named_parameters():
    if name.startswith("classifier"):
        param.require_grad = True
    else:
        param.require_grad = False

In [10]:
def format_data(data):
    data["label"] = [label2id[label] for label in data["label"]]
    tokenized = tokenizer(data["prompt"],truncation=True)
    data["token_type_ids"] = tokenized["token_type_ids"]
    data["input_ids"] = tokenized["input_ids"]
    data["attention_mask"] = tokenized["attention_mask"]
    return data

In [11]:
from datasets import Dataset

data = Dataset.from_json("training_data.json").shuffle(seed=42)
data = data.map(format_data,batched=True)
data_eval = data.select(range(30))
data = data.select(range(30,300))

In [12]:
from transformers import DataCollatorWithPadding

data_collat = DataCollatorWithPadding(tokenizer=tokenizer)

In [13]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_output",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collat,
    train_dataset=data,
    eval_dataset=data_eval,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [15]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
No log,1.127375,0,0.266667,0.262987


{'eval_loss': 1.1273750066757202,
 'eval_accuracy': 0.26666666666666666,
 'eval_f1': 0.262987012987013}

In [16]:
trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


TrainOutput(global_step=51, training_loss=0.6981944663851869, metrics={'train_runtime': 29.4608, 'train_samples_per_second': 27.494, 'train_steps_per_second': 1.731, 'total_flos': 7363064549232.0, 'train_loss': 0.6981944663851869, 'epoch': 3.0})

In [17]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.450988,51,1.000000,1.000000


{'eval_loss': 0.450987845659256, 'eval_accuracy': 1.0, 'eval_f1': 1.0}

In [18]:
import torch
def classify(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).to("cuda:0")
    
    with torch.no_grad():
        outputs = trainer.model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = torch.argmax(probs).item()
    
    return {
        "intent": id2label[predicted_id],
        "confidence": round(probs[predicted_id].item(), 4),
        "all_scores": {id2label[i]: round(p.item(), 4) for i, p in enumerate(probs)}
    }

classify("hehe remove yesterday events")
# → {"intent": "reminder", "confidence": 0.9821, "all_scores": {...}}

{'intent': 'delete',
 'confidence': 0.4623,
 'all_scores': {'planner': 0.219, 'delete': 0.4623, 'get_event': 0.3187}}

In [20]:
trainer.model.save_pretrained("bert-schduler-classification")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


In [26]:
trainer.model.push_to_hub("bert-AIPlanner-classification")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.50it/s]
Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 5.46MB/s  
New Data Upload: 100%|██████████|  364MB /  364MB, 5.31MB/s  


CommitInfo(commit_url='https://huggingface.co/sea-rod/bert-AIPlanner-classification/commit/0c28cf2eba074a1630c5471d5bc7069fefb779a4', commit_message='Upload BertForSequenceClassification', commit_description='', oid='0c28cf2eba074a1630c5471d5bc7069fefb779a4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sea-rod/bert-AIPlanner-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='sea-rod/bert-AIPlanner-classification'), pr_revision=None, pr_num=None)

In [28]:
trainer.processing_class.push_to_hub("bert-AIPlanner-classification")

CommitInfo(commit_url='https://huggingface.co/sea-rod/bert-AIPlanner-classification/commit/6c6b70ecbbd195929ae414af6daed9f871c4030a', commit_message='Upload tokenizer', commit_description='', oid='6c6b70ecbbd195929ae414af6daed9f871c4030a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sea-rod/bert-AIPlanner-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='sea-rod/bert-AIPlanner-classification'), pr_revision=None, pr_num=None)

In [24]:
from transformers import pipeline

pipe = pipeline("text-classification",model=trainer.model,tokenizer=trainer.processing_class)


In [27]:
pipe("hello")

[{'label': 'get_event', 'score': 0.5411084890365601}]